# Engagement 2: Mobile-Money Agent Data Cleaning

## Introduction

This engagement focuses on auditing, cleaning and validating mobile-money agent transaction data. The objective is to improve data quality by identifying duplicate records, inconsistent transaction types, missing and invalid amounts, date-format problems and unknown agent references.

In [1]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles available:")

print(os.listdir())

Current working directory: /content

Files available:
['.config', 'sample_data']


In [2]:
import numpy as np
import pandas as pd

# Set the original random seed
rng = np.random.default_rng(44)

# Create the agent register
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

# Create transaction records
txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

# Introduce missing and invalid amounts
idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"

# Introduce inconsistent date formats
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

# Introduce duplicate transactions
txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

# Introduce unknown agent IDs
txns.loc[
    rng.choice(len(txns), 8, replace=False), "agent_id"
] = "AG099"

# Save the original datasets
agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Agent register created:", agents.shape)
print("Transactions created:", txns.shape)
print("\nBoth CSV files have been saved successfully.")

Agent register created: (20, 3)
Transactions created: (612, 5)

Both CSV files have been saved successfully.


In [3]:
# Load the two original CSV files

agents = pd.read_csv("agents.csv")

txns = pd.read_csv("agent_transactions.csv")

# Display confirmation
print("Agent register loaded successfully!")
print("Transaction data loaded successfully!")

Agent register loaded successfully!
Transaction data loaded successfully!


In [4]:
print("AGENT REGISTER SHAPE:", agents.shape)

print("TRANSACTION DATA SHAPE:", txns.shape)

AGENT REGISTER SHAPE: (20, 3)
TRANSACTION DATA SHAPE: (612, 5)


In [5]:
txns.head()

,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [6]:
txns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB


In [7]:
print("Missing values in each column:")
print(txns.isna().sum())

Missing values in each column:
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


In [8]:
print("Total transaction rows:", len(txns))

print("Duplicate transaction IDs:",
      txns.duplicated(subset="txn_id").sum())

Total transaction rows: 612
Duplicate transaction IDs: 12


In [9]:
print("TRANSACTION TYPES AND THEIR COUNTS:")
print(txns["txn_type"].value_counts(dropna=False))

TRANSACTION TYPES AND THEIR COUNTS:
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64


In [10]:
print("AGENT IDs IN TRANSACTION DATA:")
print(txns["agent_id"].value_counts().head(10))

print("\nTransactions using AG099:")
print((txns["agent_id"] == "AG099").sum())

AGENT IDs IN TRANSACTION DATA:
agent_id
AG017    38
AG006    38
AG004    37
AG007    36
AG010    36
AG003    35
AG016    34
AG014    33
AG015    29
AG020    29
Name: count, dtype: int64

Transactions using AG099:
8


In [11]:
print("TRANSACTION DATE EXAMPLES:")
print(txns["txn_date"].tail(15))

print("\nTRANSACTION DATE DATA TYPE:")
print(txns["txn_date"].dtype)

TRANSACTION DATE EXAMPLES:
597    2026-04-03
598    2026-04-18
599    2026-03-25
600    2026-03-01
601    2026-04-03
602    2026-04-08
603    2026-03-14
604    2026-03-22
605    2026-04-28
606    2026-03-19
607    2026-03-03
608    2026-04-07
609    2026-04-18
610    2026-03-11
611    2026-04-07
Name: txn_date, dtype: object

TRANSACTION DATE DATA TYPE:
object


In [12]:
print("AMOUNT SUMMARY:")
print(txns["amount"].describe())

print("\nTRANSACTIONS WITH MISSING AMOUNTS:")
print(txns["amount"].isna().sum())

print("\nTRANSACTIONS WITH AMOUNT 250000:")
print((txns["amount"] == 250000).sum())

AMOUNT SUMMARY:
count       590.000000
mean       4537.201695
std       22836.165375
min         109.000000
25%        1086.750000
50%        1779.000000
75%        3069.750000
max      250000.000000
Name: amount, dtype: float64

TRANSACTIONS WITH MISSING AMOUNTS:
22

TRANSACTIONS WITH AMOUNT 250000:
5


## Task 10: Initial Data Quality Audit

The initial inspection identified several data quality issues in the mobile-money transaction dataset. These include missing transaction amounts, duplicate transaction IDs, inconsistent transaction type labels, suspicious transaction amounts and references to agents who are not present in the official agent register.

In [14]:
audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": txns.duplicated(subset="txn_id").sum()
    },
    {
        "issue": "Missing transaction amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": txns["amount"].isna().sum()
    },
    {
        "issue": "Inconsistent transaction type labels",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": txns["txn_type"].ne(
    txns["txn_type"].str.strip().str.title()
).sum()
    },
    {
        "issue": "Suspicious amounts of KES 250,000",
        "dimension": "Accuracy",
        "column": "amount",
        "rows_affected": (txns["amount"] == 250000).sum()
    },
    {
        "issue": "Unknown agent references",
        "dimension": "Validity",
        "column": "agent_id",
        "rows_affected": (txns["agent_id"] == "AG099").sum()
    }
])

audit

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing transaction amounts,Completeness,amount,22
2,Inconsistent transaction type labels,Consistency,txn_type,384
3,"Suspicious amounts of KES 250,000",Accuracy,amount,5
4,Unknown agent references,Validity,agent_id,8


## Task 11: Remove Duplicate Transactions

Duplicate transactions can lead to inaccurate transaction counts and financial summaries. I identified 12 duplicate transaction IDs and will retain the first occurrence of each ID while removing subsequent duplicates.

In [15]:
# Record the number of rows before removing duplicates
rows_before = len(txns)

# Remove duplicate transaction IDs, keeping the first occurrence
txns = txns.drop_duplicates(subset="txn_id", keep="first").copy()

# Record the number of rows after removing duplicates
rows_after = len(txns)

# Calculate the number of removed rows
duplicates_removed = rows_before - rows_after

print("Rows before cleaning:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Duplicate rows removed:", duplicates_removed)
print("Remaining duplicate IDs:", txns.duplicated(subset="txn_id").sum())

Rows before cleaning: 612
Rows after removing duplicates: 600
Duplicate rows removed: 12
Remaining duplicate IDs: 0


## Task 12: Standardise Transaction Types

The transaction type column contains inconsistent capitalisation. I will remove unnecessary spaces and standardise all labels to two consistent categories: Deposit and Withdrawal.

In [16]:
# Remove unnecessary spaces and standardise capitalisation
txns["txn_type"] = txns["txn_type"].str.strip().str.title()

# Check the cleaned transaction types
print("Transaction types after standardisation:")
print(txns["txn_type"].value_counts())

# Check for invalid transaction types
valid_types = ["Deposit", "Withdrawal"]

invalid_types = txns[
    ~txns["txn_type"].isin(valid_types)
]

print("\nInvalid transaction types:", len(invalid_types))

# Check for missing transaction types
print("Missing transaction types:", txns["txn_type"].isna().sum())

Transaction types after standardisation:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Invalid transaction types: 0
Missing transaction types: 0


## Task 13: Convert and Clean Transaction Amounts

The amount column contains missing values. I will convert it into a numeric data type, preserve a flag identifying imputed values and fill missing amounts using the median for each transaction type. This approach keeps the records available for analysis while making the imputation transparent.

In [17]:
# Convert amount into a numeric column
txns["amount"] = pd.to_numeric(txns["amount"], errors="coerce")

# Identify missing amounts before filling them
txns["amount_imputed"] = txns["amount"].isna()

# Calculate the median amount for each transaction type
type_medians = txns.groupby("txn_type")["amount"].median()

print("Median amount by transaction type:")
print(type_medians)

# Fill missing amounts using the relevant transaction type median
txns["amount"] = txns["amount"].fillna(
    txns.groupby("txn_type")["amount"].transform("median")
)

# Verify the results
print("\nMissing amounts after imputation:")
print(txns["amount"].isna().sum())

print("\nAmounts imputed:")
print(txns["amount_imputed"].sum())

print("\nAmount data type:")
print(txns["amount"].dtype)

Median amount by transaction type:
txn_type
Deposit       1803.5
Withdrawal    1750.5
Name: amount, dtype: float64

Missing amounts after imputation:
0

Amounts imputed:
20

Amount data type:
float64


## Task 13: Standardise Transaction Dates

The transaction date column was initially stored as text and contained inconsistent date formats. I will convert the dates into pandas datetime values, using mixed-format parsing to accommodate both hyphen-separated and slash-separated dates. Invalid dates will be identified rather than silently accepted.

In [18]:
# Convert transaction dates into proper datetime values
txns["txn_date"] = pd.to_datetime(
    txns["txn_date"],
    format="mixed",
    errors="coerce"
)

# Check the date data type
print("Transaction date data type:")
print(txns["txn_date"].dtype)

# Check for missing or invalid dates
print("\nMissing or invalid dates:")
print(txns["txn_date"].isna().sum())

# Display the earliest and latest transaction dates
print("\nEarliest transaction date:")
print(txns["txn_date"].min())

print("\nLatest transaction date:")
print(txns["txn_date"].max())

# Display sample dates
print("\nSample cleaned dates:")
print(txns["txn_date"].head(10))

Transaction date data type:
datetime64[ns]

Missing or invalid dates:
0

Earliest transaction date:
2026-03-01 00:00:00

Latest transaction date:
2026-04-29 00:00:00

Sample cleaned dates:
0   2026-03-03
1   2026-04-27
2   2026-04-29
3   2026-04-24
4   2026-04-17
5   2026-04-23
6   2026-04-02
7   2026-04-12
8   2026-04-04
9   2026-04-17
Name: txn_date, dtype: datetime64[ns]


## Task 15: Detect and Flag Outliers Using IQR

The transaction amounts contain five unusually large transactions of KES 250,000. I will use the Interquartile Range (IQR) method to identify unusually high or low values. Potential outliers will be flagged using an is_outlier column and retained for further investigation rather than automatically deleted.

In [19]:
# Calculate the first and third quartiles
Q1 = txns["amount"].quantile(0.25)
Q3 = txns["amount"].quantile(0.75)

# Calculate the interquartile range
IQR = Q3 - Q1

# Calculate the lower and upper boundaries
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# Create an outlier flag
txns["is_outlier"] = (
    (txns["amount"] < lower_bound) |
    (txns["amount"] > upper_bound)
)

# Display the results
print("First quartile (Q1):", Q1)
print("Third quartile (Q3):", Q3)
print("Interquartile range (IQR):", IQR)

print("\nLower boundary:", lower_bound)
print("Upper boundary:", upper_bound)

print("\nNumber of potential outliers:")
print(txns["is_outlier"].sum())

print("\nTransactions worth KES 250,000:")
print((txns["amount"] == 250000).sum())

print("\nFlagged transactions:")
print(txns.loc[
    txns["is_outlier"],
    ["txn_id", "txn_type", "amount", "txn_date"]
].head(15))

First quartile (Q1): 1105.0
Third quartile (Q3): 2868.5
Interquartile range (IQR): 1763.5

Lower boundary: -1540.25
Upper boundary: 5513.75

Number of potential outliers:
53

Transactions worth KES 250,000:
5

Flagged transactions:
      txn_id    txn_type  amount   txn_date
2    TX00003     Deposit  9129.0 2026-04-29
45   TX00046     Deposit  5632.0 2026-03-02
55   TX00056     Deposit  8507.0 2026-04-01
56   TX00057     Deposit  6006.0 2026-03-05
57   TX00058  Withdrawal  8035.0 2026-04-16
63   TX00064  Withdrawal  6566.0 2026-04-20
65   TX00066  Withdrawal  7116.0 2026-04-22
67   TX00068  Withdrawal  7962.0 2026-03-03
69   TX00070     Deposit  5856.0 2026-04-11
87   TX00088     Deposit  7072.0 2026-04-27
98   TX00099  Withdrawal  5643.0 2026-03-30
99   TX00100     Deposit  5777.0 2026-04-20
105  TX00106  Withdrawal  5749.0 2026-04-10
117  TX00118  Withdrawal  7465.0 2026-03-19
121  TX00122     Deposit  6310.0 2026-03-06


## Task 16: Merge Transactions with the Agent Register

I will merge the cleaned transaction data with the official agent register using agent_id as the common key. A left merge will preserve all transaction records, including those belonging to unknown agents. Unknown agent references will be identified and reported rather than silently removed.

In [20]:
# Merge transactions with the official agent register
merged_txns = txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

# Identify transactions without a matching agent
unknown_agents = merged_txns[
    merged_txns["_merge"] == "left_only"
]

print("Transactions before merge:", len(txns))
print("Transactions after merge:", len(merged_txns))

print("\nUnknown agent transactions:", len(unknown_agents))

print("\nUnknown agent IDs:")
print(unknown_agents["agent_id"].value_counts())

print("\nMerge results:")
print(merged_txns["_merge"].value_counts())

Transactions before merge: 600
Transactions after merge: 600

Unknown agent transactions: 7

Unknown agent IDs:
agent_id
AG099    7
Name: count, dtype: int64

Merge results:
_merge
both          593
left_only       7
right_only      0
Name: count, dtype: int64


## Task 17: County-Level Transaction Analysis

I will analyse mobile-money transaction activity by county. The analysis will compare total transaction value, transaction count and average transaction value, with separate summaries for deposits and withdrawals. Transactions linked to unknown agents will remain in the dataset but will not be assigned to an invented county.

In [21]:
# Keep only transactions matched to registered agents for county analysis
known_county_txns = merged_txns[
    merged_txns["county"].notna()
].copy()

# Summarise transaction activity by county and transaction type
county_summary = (
    known_county_txns
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        transaction_count=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .round(2)
    .reset_index()
)

print("COUNTY TRANSACTION SUMMARY:")
display(county_summary)

COUNTY TRANSACTION SUMMARY:


,county,txn_type,total_value,transaction_count,average_value
0,Kiambu,Deposit,334737.0,43,7784.58
1,Kiambu,Withdrawal,77024.5,27,2852.76
2,Kisumu,Deposit,436501.5,74,5898.67
3,Kisumu,Withdrawal,76348.5,37,2063.47
4,Mombasa,Deposit,149094.0,72,2070.75
5,Mombasa,Withdrawal,368592.0,61,6042.49
6,Nairobi,Deposit,543698.0,118,4607.61
7,Nairobi,Withdrawal,161932.0,62,2611.81
8,Nakuru,Deposit,184944.0,60,3082.40
9,Nakuru,Withdrawal,328568.5,39,8424.83


In [22]:
# Create a weekly transaction period
known_county_txns["week"] = (
    known_county_txns["txn_date"].dt.to_period("W").astype(str)
)

# Summarise weekly transaction values by county
weekly_summary = (
    known_county_txns
    .pivot_table(
        index="week",
        columns="county",
        values="amount",
        aggfunc="sum",
        fill_value=0
    )
)

print("WEEKLY TRANSACTION VALUE BY COUNTY:")
display(weekly_summary)

WEEKLY TRANSACTION VALUE BY COUNTY:


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru
week,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2770.5
2026-03-02/2026-03-08,18317.0,287933.0,285696.5,68967.0,56574.0
2026-03-09/2026-03-15,267745.5,23186.0,30504.5,52887.5,28730.0
2026-03-16/2026-03-22,22054.0,28126.0,25171.5,43134.5,39795.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41400.5,18972.5
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0
2026-04-06/2026-04-12,15030.0,15341.0,47119.5,56813.5,276609.0
2026-04-13/2026-04-19,12265.0,29470.0,26730.0,56406.5,26179.5
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65135.0,30862.0


In [25]:
%%writefile cleaning_pipeline.py
import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean mobile-money transaction data and merge it with the agent register.

    Returns:
        clean_df: cleaned transaction DataFrame
        log: list of data quality issues and affected row counts
    """

    # Work on copies so the original DataFrames are not modified
    clean_df = txns.copy()
    agent_df = agents.copy()

    log = []

    # 1. Record duplicate transaction IDs
    duplicate_count = clean_df.duplicated(subset="txn_id").sum()

    log.append({
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": int(duplicate_count)
    })

    # Remove duplicates, keeping the first occurrence
    clean_df = clean_df.drop_duplicates(
        subset="txn_id",
        keep="first"
    ).copy()

    # 2. Standardise transaction types
    original_types = clean_df["txn_type"].copy()

    clean_df["txn_type"] = (
        clean_df["txn_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    valid_types = ["Deposit", "Withdrawal"]

    invalid_types = ~clean_df["txn_type"].isin(valid_types)

    log.append({
        "issue": "Inconsistent or invalid transaction types",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": int(
            (original_types != clean_df["txn_type"]).sum()
        )
    })

    # 3. Convert amounts to numeric values
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"],
        errors="coerce"
    )

    missing_amounts = clean_df["amount"].isna()

    log.append({
        "issue": "Missing or invalid transaction amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": int(missing_amounts.sum())
    })

    # Create an imputation flag before filling missing values
    clean_df["amount_imputed"] = missing_amounts

    # Fill missing amounts using the median for each transaction type
    type_medians = clean_df.groupby("txn_type")["amount"].transform("median")

    overall_median = clean_df["amount"].median()

    clean_df["amount"] = clean_df["amount"].fillna(
        type_medians.fillna(overall_median)
    )

    # 4. Convert transaction dates into datetime values
    clean_df["txn_date"] = pd.to_datetime(
        clean_df["txn_date"],
        format="mixed",
        errors="coerce"
    )

    log.append({
        "issue": "Missing or invalid transaction dates",
        "dimension": "Validity",
        "column": "txn_date",
        "rows_affected": int(clean_df["txn_date"].isna().sum())
    })

    # 5. Detect outliers using the IQR method
    Q1 = clean_df["amount"].quantile(0.25)
    Q3 = clean_df["amount"].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    clean_df["is_outlier"] = (
        (clean_df["amount"] < lower_bound) |
        (clean_df["amount"] > upper_bound)
    )

    log.append({
        "issue": "Potential transaction amount outliers",
        "dimension": "Accuracy",
        "column": "amount",
        "rows_affected": int(clean_df["is_outlier"].sum())
    })

    # 6. Merge transactions with the official agent register
    clean_df = clean_df.merge(
        agent_df[["agent_id", "county"]],
        on="agent_id",
        how="left"
    )

    # Flag transactions belonging to unknown agents
    clean_df["unknown_agent"] = clean_df["county"].isna()

    log.append({
        "issue": "Unknown agent references",
        "dimension": "Validity",
        "column": "agent_id",
        "rows_affected": int(clean_df["unknown_agent"].sum())
    })

    # Return the cleaned dataset and audit log
    return clean_df, log

Overwriting cleaning_pipeline.py


In [26]:
from cleaning_pipeline import clean_agent_data

# Run the reusable cleaning function on the original data
clean_test, cleaning_log = clean_agent_data(
    pd.read_csv("agent_transactions.csv"),
    pd.read_csv("agents.csv")
)

# Display the cleaning audit log
cleaning_audit = pd.DataFrame(cleaning_log)

display(cleaning_audit)

print("\nCleaned dataset shape:", clean_test.shape)
print("Remaining duplicate IDs:",
      clean_test["txn_id"].duplicated().sum())
print("Missing amounts:", clean_test["amount"].isna().sum())
print("Missing dates:", clean_test["txn_date"].isna().sum())
print("Unknown agent transactions:",
      clean_test["unknown_agent"].sum())

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Inconsistent or invalid transaction types,Consistency,txn_type,376
2,Missing or invalid transaction amounts,Completeness,amount,20
3,Missing or invalid transaction dates,Validity,txn_date,0
4,Potential transaction amount outliers,Accuracy,amount,53
5,Unknown agent references,Validity,agent_id,7



Cleaned dataset shape: (600, 9)
Remaining duplicate IDs: 0
Missing amounts: 0
Missing dates: 0
Unknown agent transactions: 7


## Task 18: Export the Cleaned Transaction Dataset

After implementing and testing the reusable cleaning pipeline, I will export the cleaned transaction data. The exported dataset retains unknown agent transactions for investigation, includes the amount imputation and outlier flags, and contains the county information obtained from the agent register.

In [27]:
import os

# Use the output from the reusable cleaning pipeline
clean_df = clean_test.copy()

# Create the outputs folder if it does not exist
os.makedirs("outputs", exist_ok=True)

# Export the cleaned dataset
clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned dataset exported successfully!")

print("File exists:",
      os.path.exists("outputs/agent_transactions_clean.csv"))

print("Exported rows:",
      len(pd.read_csv("outputs/agent_transactions_clean.csv")))

Cleaned dataset exported successfully!
File exists: True
Exported rows: 600


## Task 19: Final Data Quality Validation

I will validate the exported transaction dataset to confirm that it meets the assignment requirements. The checks will cover duplicate transaction IDs, transaction type consistency, numeric amounts, date formatting, imputation flags, outlier flags and county information.

In [28]:
# Load the exported CSV
final_df = pd.read_csv("outputs/agent_transactions_clean.csv")

# Check the final dataset
print("FINAL DATASET SHAPE:", final_df.shape)

print("\nDUPLICATE TRANSACTION IDs:")
print(final_df["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES:")
print(final_df["txn_type"].value_counts())

print("\nAMOUNT DATA TYPE:")
print(final_df["amount"].dtype)

print("\nMISSING AMOUNTS:")
print(final_df["amount"].isna().sum())

print("\nDATE FORMAT EXAMPLE:")
print(final_df["txn_date"].head())

print("\nIMPUTED AMOUNTS:")
print(final_df["amount_imputed"].sum())

print("\nOUTLIER FLAGS:")
print(final_df["is_outlier"].sum())

print("\nUNKNOWN AGENT TRANSACTIONS:")
print(final_df["unknown_agent"].sum())

print("\nMISSING COUNTY VALUES:")
print(final_df["county"].isna().sum())

FINAL DATASET SHAPE: (600, 9)

DUPLICATE TRANSACTION IDs:
0

TRANSACTION TYPES:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

AMOUNT DATA TYPE:
float64

MISSING AMOUNTS:
0

DATE FORMAT EXAMPLE:
0    2026-03-03
1    2026-04-27
2    2026-04-29
3    2026-04-24
4    2026-04-17
Name: txn_date, dtype: object

IMPUTED AMOUNTS:
20

OUTLIER FLAGS:
53

UNKNOWN AGENT TRANSACTIONS:
7

MISSING COUNTY VALUES:
7


# Data Quality Report: Mobile-Money Agent Transactions

## 1. Overview

The original transaction dataset contained 612 records, while the official agent register contained 20 agents. After cleaning and validation, the final dataset contains 600 unique transactions. The cleaning process was implemented using a reusable Python function in `cleaning_pipeline.py`.

## 2. Data Quality Assessment

### Completeness
The original dataset contained missing transaction amounts. After duplicate removal, 20 missing amounts remained. These were imputed using the median amount for each transaction type. The final dataset contains zero missing amounts.

### Uniqueness
The original dataset contained 12 duplicate transaction IDs. Duplicate records were removed while retaining the first occurrence of each transaction ID. The final dataset contains 600 unique transactions and zero duplicate IDs.

### Consistency
Transaction type labels had inconsistent capitalisation. These were standardised into two categories: Deposit (373 transactions) and Withdrawal (227 transactions). No invalid transaction types remain.

### Validity
Transaction dates were successfully converted into datetime values, covering 1 March to 29 April 2026. Seven transactions referenced an unknown agent, AG099. These records were retained and flagged rather than deleted.

### Accuracy
The IQR method identified 53 potential outliers. These transactions were retained and flagged using the `is_outlier` column for further investigation. The `amount_imputed` column identifies the 20 estimated transaction amounts.

### Timeliness
Transaction dates cover the period from 1 March 2026 to 29 April 2026. Dates were standardised to the ISO format YYYY-MM-DD to support reliable chronological analysis and reporting.

## 3. Recommendations

- Investigate the seven transactions associated with unregistered agent AG099.
- Verify the 53 flagged outlier transactions before making financial decisions.
- Review the 20 imputed amounts against original transaction records where possible.
- Enforce standard transaction type labels during data entry.
- Maintain regular data quality checks and updates to the official agent register.

## 4. Conclusion

The cleaning process improved the completeness, uniqueness, consistency, validity and reliability of the transaction dataset. The final dataset contains 600 unique records, numeric transaction amounts, standardised dates, county information where available, and transparent imputation and outlier flags. The reusable cleaning pipeline makes the process repeatable and easier to maintain.

In [29]:
%%writefile generate_data.py
import numpy as np
import pandas as pd

# ==========================================
# ENGAGEMENT 1: DUKA ELECTRONICS SALES DATA
# ==========================================

rng = np.random.default_rng(2026)

n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(
                      rng.integers(0, 180, n), unit="D"
                  ),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)


# ==========================================
# ENGAGEMENT 2: MOBILE-MONEY AGENT DATA
# ==========================================

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),
    "onboarded": pd.date_range(
        "2024-01-01",
        periods=20,
        freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"],
        n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(
            rng.integers(0, 60, n), unit="D"
        )
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("All datasets generated successfully!")
print("Duka sales:", sales.shape)
print("Agent register:", agents.shape)
print("Agent transactions:", txns.shape)

Writing generate_data.py


In [30]:
import os

print("Generator file exists:", os.path.exists("generate_data.py"))
print("File size:", os.path.getsize("generate_data.py"), "bytes")

Generator file exists: True
File size: 2666 bytes


In [31]:
!python generate_data.py

All datasets generated successfully!
Duka sales: (1000, 7)
Agent register: (20, 3)
Agent transactions: (612, 5)


In [32]:
import pandas as pd
import os

for filename in ["duka_sales.csv", "agents.csv", "agent_transactions.csv"]:
    print(filename, "exists:", os.path.exists(filename))

print("\nDataset shapes:")
print("Duka sales:", pd.read_csv("duka_sales.csv").shape)
print("Agent register:", pd.read_csv("agents.csv").shape)
print("Agent transactions:", pd.read_csv("agent_transactions.csv").shape)

duka_sales.csv exists: True
agents.csv exists: True
agent_transactions.csv exists: True

Dataset shapes:
Duka sales: (1000, 7)
Agent register: (20, 3)
Agent transactions: (612, 5)


In [33]:
%%writefile requirements.txt
pandas
numpy

Writing requirements.txt


In [34]:
with open("requirements.txt", "r") as file:
    print(file.read())

pandas
numpy



In [35]:
import os

print("PROJECT FILES CHECK")
print("===================")

required_files = [
    "generate_data.py",
    "cleaning_pipeline.py",
    "requirements.txt",
    "outputs/agent_transactions_clean.csv",
    "outputs/premium_online_coast_capital.csv",
    "excel/duka_branch_summary.xlsx"
]

for filename in required_files:
    print(filename, ":", "FOUND" if os.path.exists(filename) else "MISSING")

PROJECT FILES CHECK
generate_data.py : FOUND
cleaning_pipeline.py : FOUND
requirements.txt : FOUND
outputs/agent_transactions_clean.csv : FOUND
outputs/premium_online_coast_capital.csv : MISSING
excel/duka_branch_summary.xlsx : MISSING


In [36]:
from google.colab import files

uploaded_files = files.upload()

Saving duka_branch_summary.xlsx to duka_branch_summary.xlsx
Saving premium_online_coast_capital.csv to premium_online_coast_capital.csv


In [37]:
import os
import shutil

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

shutil.move(
    "premium_online_coast_capital.csv",
    "outputs/premium_online_coast_capital.csv"
)

shutil.move(
    "duka_branch_summary.xlsx",
    "excel/duka_branch_summary.xlsx"
)

print("Both files moved successfully!")

Both files moved successfully!


In [38]:
required_files = [
    "generate_data.py",
    "cleaning_pipeline.py",
    "requirements.txt",
    "outputs/agent_transactions_clean.csv",
    "outputs/premium_online_coast_capital.csv",
    "excel/duka_branch_summary.xlsx"
]

for filename in required_files:
    print(filename, ":", "FOUND" if os.path.exists(filename) else "MISSING")

generate_data.py : FOUND
cleaning_pipeline.py : FOUND
requirements.txt : FOUND
outputs/agent_transactions_clean.csv : FOUND
outputs/premium_online_coast_capital.csv : FOUND
excel/duka_branch_summary.xlsx : FOUND
